In [1]:
# Step1: As the first step, I import Python libraries.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import os

In [2]:
# Step2: The data source for this project is an Excel file.

electricity_consumption_table = pd.read_csv('/content/Electric_Production.csv')


In [3]:
# Step3: Let's check dimensions of this table by using 'shape' attribute:

electricity_consumption_table.shape

(397, 2)

In [ ]:
# Our dataset has 397 rows and 2 columns.

In [4]:
# Step4: Let's check datatypes by using info() method:

electricity_consumption_table.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DATE    397 non-null    object 
 1   Value   397 non-null    float64
dtypes: float64(1), object(1)
memory usage: 6.3+ KB


In [ ]:
# There are 2 columns in this dataset: 1)DATE and 2)Value
# DATE column has type of "object" which in this case, it is string. Later, we need to convert the type to datetime.
# Value column has type of "float64"

In [5]:
# Step5: It looks there are no NULL values, but we double check that with .isnull() as well:

electricity_consumption_table.isnull().sum()

,0
DATE,0
Value,0


In [ ]:
# The data does not have any NULL values.


In [6]:
# Step6: And now let's check out first few rows of our dataset:

electricity_consumption_table.head()

,DATE,Value
0,01-01-1985,72.5052
1,02-01-1985,70.6720
2,03-01-1985,62.4502
3,04-01-1985,57.4714
4,05-01-1985,55.3151


In [ ]:
# From the table shown above, I can see measurements are recorded on the first day of every month.
# Therefore we have 1 record per month, which means 12 records per year.

In [7]:
# Step7: The data history starts from January 1st 1985. Let's check the endpoint of this data by using tail() method:

electricity_consumption_table.tail()


,DATE,Value
392,09-01-2017,98.6154
393,10-01-2017,93.6137
394,11-01-2017,97.3359
395,12-01-2017,114.7212
396,01-01-2018,129.4048


In [ ]:
# So the data is from January 1st 1985 to January 1st 2018.
# The table shape looks good and we do not need to make a transpose of it. However, it is a good idea to add a column as index to left side of this table

In [8]:
# Step8: renaming index column on the left side to 'Index' by using .index.name:

electricity_consumption_table.index.name = 'Index'

In [9]:
# Step9: Rename the dataset to df:
df = electricity_consumption_table

In [10]:
# Step10: Check the first 10 rows with head(10):
df.head(10)

,DATE,Value
Index,,
0,01-01-1985,72.5052
1,02-01-1985,70.6720
2,03-01-1985,62.4502
3,04-01-1985,57.4714
4,05-01-1985,55.3151
5,06-01-1985,58.0904
6,07-01-1985,62.6202
7,08-01-1985,63.2485
8,09-01-1985,60.5846


In [11]:
# Step11: Now, let's change DATE column datatype to datetime, which also includes changing of date format to YYYY-MM-DD.

df['DATE'] = pd.to_datetime(df['DATE'])


In [12]:
# Step12: Also, let's round-up our float values at Value column to 2 decimal points.

df['Value'] = df['Value'].round(2)


In [13]:
# Step13: I assume consumption values are in kWh unit. So, let's rename 'DATE' column to 'Date' and 'Value' column to 'Consumption (kWh)':

df = df.rename(columns={'DATE':'Date', 'Value':'Consumption (kWh)'})

In [14]:
# Step14: Now, we use describe() to get some useful statistical information:

df.describe()

,Date,Consumption (kWh)
count,397,397.000000
mean,2001-07-01 13:25:14.357682560,88.847229
min,1985-01-01 00:00:00,55.320000
25%,1993-04-01 00:00:00,77.110000
50%,2001-07-01 00:00:00,89.780000
75%,2009-10-01 00:00:00,100.520000
max,2018-01-01 00:00:00,129.400000
std,NaN,15.387785


In [ ]:
# Summary:
# Total number of records = 397
# First measurement was recorded on Jan 1st 1985
# Last measurement was recorded on Jan 1st 2018
# Maximum consumption = 129.4
# Minimum consumption = 55.32
# Average consumption = 88.84

In [15]:
# Step15: Let's sort Consumption values column to see if we can get any clue at this stage:

df['Consumption (kWh)'].sort_values(ascending = False)

,Consumption (kWh)
Index,
396,129.40
348,124.25
360,120.27
312,119.49
300,119.02
...,...
3,57.47
15,57.03
9,56.32


In [ ]:
# The table above shows electricity consumption is higher as we get closer to 2018 and lower as we get closer to 1985. Population growth could be the reason for this increase.

In [16]:
# Step16: Now, I am adding a new column to the dataframe and call it Positive_Growth to check if a single value at Consumption column is larger than its previous one.
# I use .diff() method for this task.
# True means consumption for that month is higher than consumption for the previous month.
# False means consumption for that month is less than consumption for the previous month.
# Then I use .head(36) to see values for 3 years, or 36 months.

df['Positive_Growth'] = df['Consumption (kWh)'].diff() > 0
df.head(36)


,Date,Consumption (kWh),Positive_Growth
Index,,,
0,1985-01-01,72.51,False
1,1985-02-01,70.67,False
2,1985-03-01,62.45,False
3,1985-04-01,57.47,False
4,1985-05-01,55.32,False
5,1985-06-01,58.09,True
6,1985-07-01,62.62,True
7,1985-08-01,63.25,True
8,1985-09-01,60.58,False


In [ ]:
# I would like to add a new column to the dataframe and call it Month. Values at this column shows month names in alphabet.
# For this purpose, I make a dictionary which keys are numbers between 1 to 12, and values are correspondent month names.
# This dictionary is called month_dict. Then I use map() to map each month to the associated extracted month value from the datetime objects at Date column.

In [17]:
# Step17: Creating month_dict dictionary:

month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


In [18]:
# Step18: Creating 'Month' column in our dataframe and then map each month from Date column to associated month name at month_dict dictionary.

df['Month'] = df['Date'].dt.month.map(month_dict)
df.head(12)

,Date,Consumption (kWh),Positive_Growth,Month
Index,,,,
0,1985-01-01,72.51,False,January
1,1985-02-01,70.67,False,February
2,1985-03-01,62.45,False,March
3,1985-04-01,57.47,False,April
4,1985-05-01,55.32,False,May
5,1985-06-01,58.09,True,June
6,1985-07-01,62.62,True,July
7,1985-08-01,63.25,True,August
8,1985-09-01,60.58,False,September


In [20]:
# Step19: This table looks good for future Data Science works.
# Now, we save it for future works.

df.to_csv('Electicity_Consumption_df.csv')